##### Ingesting Dataset

In [ ]:
"""import os"""
import csv
from pathlib import Path
import chardet
import pandas as pd

DATA_DIR = Path("D:\email-parser-pipeline\data\hotels")
DATA_DIR.exists(), DATA_DIR.resolve()

<>:8: SyntaxWarning: invalid escape sequence '\e'
<>:8: SyntaxWarning: invalid escape sequence '\e'
C:\Users\pksju\AppData\Local\Temp\ipykernel_40544\3366838323.py:8: SyntaxWarning: invalid escape sequence '\e'
  DATA_DIR = Path("D:\email-parser-pipeline\data\hotels")


(True, WindowsPath('D:/email-parser-pipeline/data/hotels'))

In [ ]:
type(DATA_DIR)

pathlib._local.WindowsPath

In [27]:
## Step_1 --> Retreiving files as name of cities that exist
city_folders = [p for p in DATA_DIR.iterdir() if p.is_dir()]
len(city_folders)


10

In [18]:
city_folders

[WindowsPath('D:/email-parser-pipeline/data/hotels/beijing'),
 WindowsPath('D:/email-parser-pipeline/data/hotels/chicago'),
 WindowsPath('D:/email-parser-pipeline/data/hotels/dubai'),
 WindowsPath('D:/email-parser-pipeline/data/hotels/las-vegas'),
 WindowsPath('D:/email-parser-pipeline/data/hotels/london'),
 WindowsPath('D:/email-parser-pipeline/data/hotels/montreal'),
 WindowsPath('D:/email-parser-pipeline/data/hotels/new-delhi'),
 WindowsPath('D:/email-parser-pipeline/data/hotels/new-york-city'),
 WindowsPath('D:/email-parser-pipeline/data/hotels/san-francisco'),
 WindowsPath('D:/email-parser-pipeline/data/hotels/shanghai')]

In [20]:
print(f"Number of city folders found: {len(city_folders)}")
for i,c in enumerate(city_folders):
    n_files = len(list(c.glob('*')))
    print(f"{i+1})  {c.name}: {n_files} files")

Number of city folders found: 10
1)  beijing: 136 files
2)  chicago: 138 files
3)  dubai: 236 files
4)  las-vegas: 207 files
5)  london: 875 files
6)  montreal: 248 files
7)  new-delhi: 130 files
8)  new-york-city: 258 files
9)  san-francisco: 214 files
10)  shanghai: 126 files


In [ ]:
# sample check
sample_city = city_folders[0]
sample_file = list(sample_city.glob('*'))[0] # taking the first file from sample city
print(f"City: {sample_city.name}")
print(f"File: {sample_file.name}")

with open(sample_file, 'rb') as f:
    raw_bytes = f.read(500)
print(raw_bytes)

City: beijing
File: china_beijing_aloft_beijing_haidian
b"Oct 12 2009 \tNice trendy hotel location not too bad.\tI stayed in this hotel for one night. As this is a fairly new place some of the taxi drivers did not know where it was and/or did not want to drive there. Once I have eventually arrived at the hotel, I was very pleasantly surprised with the decor of the lobby/ground floor area. It was very stylish and modern. I found the reception's staff geeting me with 'Aloha' a bit out of place, but I guess they are briefed to say that to keep up the coropo"


In [28]:
with open(sample_file, 'rb') as f:
    raw = f.read(5000)
detected = chardet.detect(raw)
print(detected)

{'encoding': 'Windows-1252', 'confidence': 0.7638143075172426, 'language': 'en', 'mime_type': 'text/plain'}


In [44]:
encoding = detected['encoding'] or 'utf-8'
rows = []
bad_lines = []

with open(sample_file, 'r', encoding=encoding, errors='replace') as f:
    reader = csv.reader(f, delimiter='\t')
    for line_num, fields in enumerate(reader, start=1):
        if len(fields) == 4 and fields[-1] == '':
            fields = fields[:3]

        if len(fields) != 3:
            bad_lines.append((line_num, fields))
            continue
        rows.append(fields)

print(f"Good rows: {len(rows)}")
print(f"Bad/malformed rows: {len(bad_lines)}")
bad_lines[:5]


Good rows: 7
Bad/malformed rows: 0


[]

In [45]:
df_sample = pd.DataFrame(rows, columns=['date_raw', 'title_raw', 'review_raw'])
df_sample['city'] = sample_city.name
df_sample['hotel_name'] = sample_file.stem

df_sample.head(10)

,date_raw,title_raw,review_raw,city,hotel_name
0,Oct 12 2009,Nice trendy hotel location not too bad.,I stayed in this hotel for one night. As this ...,beijing,china_beijing_aloft_beijing_haidian
1,Sep 25 2009,Great Budget Hotel!,Stayed two nights at Aloft on the most recent ...,beijing,china_beijing_aloft_beijing_haidian
2,Aug 4 2009,Excellent value - location not a big problem.,We stayed at the Aloft Beijing Haidian for 5 n...,beijing,china_beijing_aloft_beijing_haidian
3,Jul 17 2009,Stylish clean reasonable value poor location,I am glad to be the first person to post photo...,beijing,china_beijing_aloft_beijing_haidian
4,May 30 2009,Remote but excellent value for money,Stayed there for one night. The hotel is locat...,beijing,china_beijing_aloft_beijing_haidian
5,Dec 31 2008,Good value but not downtown,This hotel is located next to the Four Points ...,beijing,china_beijing_aloft_beijing_haidian
6,Jul 20 2009,???????????????????,,beijing,china_beijing_aloft_beijing_haidian


In [46]:
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date_raw    7 non-null      str  
 1   title_raw   7 non-null      str  
 2   review_raw  7 non-null      str  
 3   city        7 non-null      str  
 4   hotel_name  7 non-null      str  
dtypes: str(5)
memory usage: 412.0 bytes


In [47]:
# Check variety in date formats — this will inform the cleaning step later
df_sample['date_raw'].sample(min(10, len(df_sample))).tolist()

['Dec 31 2008 ',
 'Aug 4 2009 ',
 'Oct 12 2009 ',
 'Jul 17 2009 ',
 'Jul 20 2009 ',
 'Sep 25 2009 ',
 'May 30 2009 ']

In [ ]:
def parse_hotel_file(file_path: Path, city: str, hotel_name: str):
    with open(file_path, 'rb') as f:
        raw = f.read(5000)
    encoding = chardet.detect(raw)['encoding'] or 'utf-8'

    parsed_rows = []
    with open(file_path, 'r', encoding=encoding, errors='replace') as f:
        reader = csv.reader(f, delimiter='\t')
        for fields in reader:
            # Same trailing-empty-field fix as the exploration cell above
            if len(fields) == 4 and fields[-1] == '':
                fields = fields[:3]

            if len(fields) != 3:
                continue
            date_raw, title_raw, review_raw = fields
            parsed_rows.append({
                'city': city,
                'hotel_name': hotel_name,
                'date_raw': date_raw.strip(),
                'title_raw': title_raw.strip(),
                'review_raw': review_raw.strip(),
            })
    return parsed_rows

# Try on a handful of files (e.g. first 3 files in first 2 cities) before running on everything
test_rows = []
for city_folder in city_folders[:2]:
    for hotel_file in list(city_folder.glob('*'))[:3]:
        test_rows.extend(parse_hotel_file(hotel_file, city_folder.name, hotel_file.stem))

df_test = pd.DataFrame(test_rows)
print(f"Total rows from test batch: {len(df_test)}")
df_test.head()


Total rows from test batch: 752


,city,hotel_name,date_raw,title_raw,review_raw
0,beijing,china_beijing_aloft_beijing_haidian,Oct 12 2009,Nice trendy hotel location not too bad.,I stayed in this hotel for one night. As this ...
1,beijing,china_beijing_aloft_beijing_haidian,Sep 25 2009,Great Budget Hotel!,Stayed two nights at Aloft on the most recent ...
2,beijing,china_beijing_aloft_beijing_haidian,Aug 4 2009,Excellent value - location not a big problem.,We stayed at the Aloft Beijing Haidian for 5 n...
3,beijing,china_beijing_aloft_beijing_haidian,Jul 17 2009,Stylish clean reasonable value poor location,I am glad to be the first person to post photo...
4,beijing,china_beijing_aloft_beijing_haidian,May 30 2009,Remote but excellent value for money,Stayed there for one night. The hotel is locat...
